<a href="https://colab.research.google.com/github/ElionLAB/OOP_2026_Practice/blob/main/ch_06/src/part_1/answer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Environment Setup — Auto-detect Google Colab / Local
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    pass  # This notebook does not require additional packages.
else:
    print('Local environment: Make sure `conda activate oop_practice` is active.')

# Lecture 6 — Part 1 (Slides 1–41): Built-ins, Callables, Context Managers & Functional Composition

**Konkuk University OOP (Python Object-Oriented Programming) — Spring 2026**

---

## Learning Objectives

1. See how built-in functions (`len`, `reversed`, `enumerate`) delegate to dunder methods.
2. Customize iteration by overriding `__reversed__`, and observe the `__len__ + __getitem__` fallback.
3. Replace "method overloading" with **default values, `*args`, `**kwargs`** — and avoid the *mutable default trap*.
4. Use `*` / `**` to **unpack** sequences and dicts at the call site.
5. Treat functions as **first-class objects** — assign attributes, pass them as callbacks, monkey patch.
6. Build a **stateful callable** with `__call__`.
7. Manage external state safely with `with`, and write your own `__enter__` / `__exit__`.
8. Re-implement Lecture 5's K-NN train/test split using **functional composition** with higher-order `partition`.

## How this notebook is structured

Each section follows the same rhythm:

1. **Concept** — short explanation, taken directly from the slides.
2. **Full code reference** — the *goal* code block (your target).
3. **TODO** cell — fill the small blanks marked `# TODO`.
4. **Quick check** — assertions verify your solution.

The TODO blanks are intentionally tiny — usually one expression per line — so you can focus on understanding *each* line, not on writing big code from scratch.

> **Continuity with Lecture 5.** Section 8 returns to the K-NN train/test split from Lecture 5 §7, but rebuilds it with a *functional* lens: free functions for the rule, a higher-order `partition` for the loop. The final cell shows that the OOP and functional versions produce the same split.

## 1. Built-in Functions ↔ Dunder Methods (Slides 2–7)

> *"In Python, functions are objects. Many functional techniques are actually executing object-oriented 'dunder' (magic) methods under the hood."* — Slide 2

Built-in functions like `len()`, `reversed()`, `enumerate()` are *global, functional interfaces* to behaviors that live as **special methods** on each object. Any class that implements the right dunder is automatically picked up — no inheritance required (duck typing).

| Built-in       | Delegates to                       |
| -------------- | ---------------------------------- |
| `len(obj)`     | `obj.__len__()`                    |
| `reversed(obj)`| `obj.__reversed__()` *or fallback* |
| `enumerate(it)`| `it.__iter__()` (yields tuples)    |

### 1.1 — `len()` and `__len__` (Slide 4)

`len(obj)` is *not* `obj.length` — Python keeps the question "how big is this?" as a global function so every object answers the same way.

### Full code reference (Slide 4)

```python
# Custom class integrating with built-in len()
class CustomSequence:
    def __init__(self, data):
        self.data = data

    def __len__(self):
        # Custom logic to define "length"
        return len(self.data)


# Usage
my_seq = CustomSequence([1, 2, 3, 4])
print(len(my_seq))   # → 4
```

In [ ]:
# TODO 1.1 — Implement CustomSequence with __len__

class CustomSequence:
    def __init__(self, data):
        # TODO 1.1-a: store `data` on self
        self.data = data

    def __len__(self):
        # TODO 1.1-b: return the length of self.data
        # Hint: use the built-in len() — which itself dispatches to list.__len__
        return len(self.data)


In [ ]:
# Quick check
my_seq = CustomSequence([1, 2, 3, 4])
assert len(my_seq) == 4, f"expected 4, got {len(my_seq)}"
print(f"OK: len(my_seq) = {len(my_seq)}")

empty = CustomSequence("")
assert len(empty) == 0
print("OK: empty sequence has length 0")
print("Section 1.1 passed.")

### 1.2 — `reversed()` and the `__len__ + __getitem__` Fallback (Slides 5–6)

When you call `reversed(obj)`, Python takes a 3-step path:

1. Call `reversed(obj)`.
2. If `obj` defines `__reversed__()` — use it.
3. Otherwise, **fall back** to `__len__()` + `__getitem__()` and iterate `i = len-1, len-2, …, 0`.

This is duck typing in action: a class that *looks like* a sequence (has `__len__` and `__getitem__`) gets `reversed()` for free.

### Full code reference (Slide 6)

```python
class CardDeck:
    def __init__(self, cards):
        self.cards = cards
    def __len__(self):       return len(self.cards)
    def __getitem__(self, i): return self.cards[i]

# No __reversed__ defined — Python uses the fallback
deck = CardDeck(['A', 'K', 'Q', 'J'])
print(list(reversed(deck)))   # ['J', 'Q', 'K', 'A']
```

In [ ]:
# TODO 1.2 — CardDeck without __reversed__ (uses the fallback)

class CardDeck:
    def __init__(self, cards):
        # TODO 1.2-a: store cards on self
        self.cards = cards

    def __len__(self):
        # TODO 1.2-b: return len(self.cards)
        return len(self.cards)

    def __getitem__(self, i):
        # TODO 1.2-c: return self.cards[i]
        return self.cards[i]
    # NOTE: deliberately NO __reversed__ — Python will build it from __len__ + __getitem__


In [ ]:
# Quick check
deck = CardDeck(['A', 'K', 'Q', 'J'])
assert list(reversed(deck)) == ['J', 'Q', 'K', 'A']
print(f"OK: reversed(deck) → {list(reversed(deck))}")

# Confirm the fallback path: CardDeck has no __reversed__
assert not hasattr(CardDeck, '__reversed__'), "fallback only kicks in when __reversed__ is absent"
print("OK: CardDeck has no __reversed__ — Python used the __len__ + __getitem__ fallback")
print("Section 1.2 passed.")

### 1.3 — Overriding `__reversed__` (Slide 7)

If you *do* define `__reversed__`, Python uses your version and **skips** the fallback. The functional interface (`reversed(obj)`) is the same — but the OOP method underneath is now yours.

### Full code reference (Slide 7)

```python
# Overriding the default reverse behavior
class FunkyBackwards(list):
    def __reversed__(self):
        # Overriding the default list reversal
        return "BACKWARDS!"
```

Calling `reversed(funkadelic)` will return the string `"BACKWARDS!"` instead of an iterator — proving the functional interface (`reversed`) routes to *your* OOP method.

In [ ]:
# TODO 1.3 — FunkyBackwards: override __reversed__

class FunkyBackwards(list):
    def __reversed__(self):
        # TODO 1.3-a: return the literal string "BACKWARDS!"
        return "BACKWARDS!"


In [ ]:
# Quick check
funkadelic = FunkyBackwards([5, 6, 7, 8, 9])
result = reversed(funkadelic)
assert result == "BACKWARDS!", f"expected 'BACKWARDS!', got {result!r}"
print(f"OK: reversed(funkadelic) → {result!r}")
print("OK: the functional interface (reversed) routed to our overridden __reversed__")
print("Section 1.3 passed.")

## 2. `enumerate()` — Index + Item in One Step (Slide 8)

### Before (manual counter)
```python
index = 0
for item in items:
    ...
    index += 1
```

### After (`enumerate`)
```python
for index, item in enumerate(items, start=1):
    ...
```

`enumerate` yields `(index, item)` tuples — *no manual integer tracking*. The `start=` keyword controls the first index.

### Full code reference (Slide 8)

```python
# Reading lines with their line numbers
with open("sample_data.txt") as file:
    # enumerate yields (index, item); start=1
    for index, line in enumerate(file, start=1):
        print(f"{index:3d}: {line.rstrip()}")
```

In [ ]:
# Setup: write a small sample file we can read back.
from pathlib import Path

Path("sample_data.txt").write_text(
    "first line\nsecond line\nthird line\n",
    encoding="utf-8",
)
print("Wrote sample_data.txt")

In [ ]:
# TODO 2 — Read sample_data.txt with enumerate(start=1)

with open("sample_data.txt") as file:
    # TODO 2-a: iterate with enumerate(file, start=1) so the first index is 1
    for index, line in enumerate(file, start=1):
        # TODO 2-b: print f"{index:3d}: {line.rstrip()}"
        print(f"{index:3d}: {line.rstrip()}")


In [ ]:
# Quick check
# Re-run the same logic but capture into a list, so we can assert.
with open("sample_data.txt") as file:
    pairs = [(i, line.rstrip()) for i, line in enumerate(file, start=1)]

assert pairs == [(1, "first line"), (2, "second line"), (3, "third line")], pairs
print(f"OK: collected {pairs}")
print("Section 2 passed.")

## 3. No Method Overloading — The Pythonic Way (Slides 9–13)

> **The Challenge** — Java/C++ allow `add(int x)` and `add(String x)` side-by-side. Python does **not** support signature-based overloading.

> **The Pythonic Solution** — one flexible function with **default values**, `*args`, and `**kwargs`.

### 3.1 — Single Method, Multiple Behaviors (Slide 10)

One function, three valid call shapes — chosen via default values + an explicit branch.

### Full code reference (Slide 10)

```python
# In Python, we use ONE flexible function
def add(a, b=0, c=None):
    if c is not None:
        return a + b + c
    return a + b


# All these calls are valid:
print(add(5))           # Output: 5
print(add(5, 10))       # Output: 15
print(add(5, 10, 15))   # Output: 30
```

In [ ]:
# TODO 3.1 — Implement `add` with default values

def add(a, b=0, c=None):
    # TODO 3.1-a: if c is not None, return a + b + c
    if c is not None:
        return a + b + c
    # TODO 3.1-b: otherwise return a + b
    return a + b


In [ ]:
# Quick check
assert add(5) == 5
assert add(5, 10) == 15
assert add(5, 10, 15) == 30
print(f"OK: add(5) = {add(5)}, add(5,10) = {add(5,10)}, add(5,10,15) = {add(5,10,15)}")
print("Section 3.1 passed.")

### 3.2 — Default Arguments + Keyword Calls (Slide 11)

Defaults make arguments *optional*. **Keywords let callers reorder them** — `mandatory_and_optional(y=3, x=2, z=4)` is fine even though `x` came first in the definition.

### Full code reference (Slide 11)

```python
# 'x','y' mandatory; 'z' optional
def mandatory_and_optional(x, y, z=5):
    return x * y * z


print(mandatory_and_optional(2, 3))       # z=5  → 30
print(mandatory_and_optional(2, 3, 10))   #      → 60

# Keywords bypass order:
print(mandatory_and_optional(y=3, x=2, z=4))   # → 24
```

In [ ]:
# TODO 3.2 — mandatory_and_optional

def mandatory_and_optional(x, y, z=5):
    # TODO 3.2-a: return the product x * y * z
    return x * y * z


In [ ]:
# Quick check
assert mandatory_and_optional(2, 3) == 30
assert mandatory_and_optional(2, 3, 10) == 60
assert mandatory_and_optional(y=3, x=2, z=4) == 24
print(f"OK: mandatory_and_optional(2,3) = {mandatory_and_optional(2,3)}")
print(f"OK: mandatory_and_optional(2,3,10) = {mandatory_and_optional(2,3,10)}")
print(f"OK: mandatory_and_optional(y=3, x=2, z=4) = {mandatory_and_optional(y=3, x=2, z=4)}")
print("Section 3.2 passed.")

### 3.3 — The Mutable Default Argument Trap (Slides 12–13)

> **Default arguments are evaluated *only once*, when the function is defined — not every time it is called.**

If the default is a *mutable* object (`[]`, `{}`), every call **shares the same instance**. Mutating it in one call leaks into the next.

### The cure (slide 13)

Use `None` as the sentinel default, then build a fresh container *inside* the function body.

### Full code reference (Slide 13)

```python
# WRONG: list is created once and reused
def bad_append(item, target_list=[]):
    target_list.append(item)
    return target_list

# CORRECT: use None and initialize inside
def good_append(item, target_list=None):
    if target_list is None:
        target_list = []
    target_list.append(item)
    return target_list
```

In [ ]:
# TODO 3.3 — Implement BOTH bad_append (the trap) and good_append (the fix)

def bad_append(item, target_list=[]):       # ⚠️ deliberately buggy default
    # TODO 3.3-a: append `item` to target_list
    target_list.append(item)
    # TODO 3.3-b: return target_list
    return target_list


def good_append(item, target_list=None):
    # TODO 3.3-c: if target_list is None, replace it with a fresh empty list
    if target_list is None:
        target_list = []
    # TODO 3.3-d: append `item` and return the list
    target_list.append(item)
    return target_list


In [ ]:
# Quick check — observe the trap, then confirm the fix.

# bad_append: per slide 13, printing AT call time exposes the bug visually.
print(f"bad_append('a') → {bad_append('a')}")    # ['a']
print(f"bad_append('b') → {bad_append('b')}")    # ['a', 'b']  ← Whoops! 'a' leaked

# Why? The default list is created ONCE, at function-definition time, and is
# then shared across every call. We can see it directly:
shared = bad_append.__defaults__[0]
assert shared == ['a', 'b'], f"the shared default should now hold both items, got {shared}"
print(f"Shared default list (bad_append.__defaults__[0]) = {shared}")

# good_append: each call gets a fresh list — the cure works.
print(f"good_append('a') → {good_append('a')}")  # ['a']
print(f"good_append('b') → {good_append('b')}")  # ['b']

g1 = good_append('a')
g2 = good_append('b')
assert g1 == ['a']
assert g2 == ['b']
assert g1 is not g2, "each good_append call must produce a NEW list"

print("Section 3.3 passed.")


## 4. Variable Arguments — `*args`, `**kwargs`, and Unpacking (Slides 14–17)

### 4.1 — `*args` Packs Positional Arguments into a Tuple (Slide 14)

Prefix a parameter with `*` to **collect** all remaining positional arguments into a tuple.

### Full code reference (Slide 14)

```python
# Accepts arbitrary numbers of URLs
def get_pages(*links):
    # 'links' is automatically a tuple
    for link in links:
        print(f"Downloading: {link}")


# All of these are valid:
get_pages()
get_pages("http://example.com")
get_pages("http://s1.com", "http://s2.com")
```

In [ ]:
# TODO 4.1 — get_pages(*links)

def get_pages(*links):
    # TODO 4.1-a: iterate over links and print f"Downloading: {link}" for each
    for link in links:
        print(f"Downloading: {link}")


In [ ]:
# Quick check — capture stdout to verify
import io, contextlib

buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    get_pages()                                       # zero args is fine
    get_pages("http://example.com")
    get_pages("http://s1.com", "http://s2.com")

out = buf.getvalue().splitlines()
assert out == [
    "Downloading: http://example.com",
    "Downloading: http://s1.com",
    "Downloading: http://s2.com",
], out
print("OK: captured downloads:")
for line in out:
    print("   ", line)
print("Section 4.1 passed.")

### 4.2 — `**kwargs` Packs Keyword Arguments into a Dict (Slide 15)

Double-star `**` collects keyword arguments into a **dictionary**. Useful for configuration setups and forwarding unknown options to `super().__init__(**kwargs)`.

### Full code reference (Slide 15)

```python
# Keywords collected in a dictionary
def connect_to_server(host, **kwargs):
    print(f"Connecting to {host}...")
    for key, value in kwargs.items():
        print(f"  - {key}: {value}")


# 'port', 'secure' become dict keys
connect_to_server("localhost", port=8080, secure=True)
```

In [ ]:
# TODO 4.2 — connect_to_server(host, **kwargs)

def connect_to_server(host, **kwargs):
    # TODO 4.2-a: print f"Connecting to {host}..."
    print(f"Connecting to {host}...")
    # TODO 4.2-b: iterate kwargs.items() and print f"  - {key}: {value}"
    for key, value in kwargs.items():
        print(f"  - {key}: {value}")


In [ ]:
# Quick check
import io, contextlib

buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    connect_to_server("localhost", port=8080, secure=True)

out = buf.getvalue().splitlines()
assert out[0] == "Connecting to localhost..."
assert "  - port: 8080" in out
assert "  - secure: True" in out
print("OK: connect_to_server output:")
for line in out:
    print("   ", line)
print("Section 4.2 passed.")

### 4.3 — Unpacking at the Call Site with `*` and `**` (Slides 16–17)

The same `*` and `**` operators work at the **call site**:

| Call form          | Effect                                              |
| ------------------ | --------------------------------------------------- |
| `f(*seq)`          | unpack a sequence into **positional** arguments     |
| `f(**dct)`         | unpack a dict into **keyword** arguments            |

This replaces clunky `f(seq[0], seq[1], seq[2])` patterns.

### Full code reference (Slide 17)

```python
# List and dictionary unpacking in a function call
def show_args(arg1, arg2, arg3="THREE"):
    print(arg1, arg2, arg3)


some_list = [0, 1, 2]
more_args = {"arg1": "ONE", "arg2": "TWO"}

# Unpacking a sequence into positional arguments
show_args(*some_list)     # Output: 0 1 2

# Unpacking a dictionary into keyword arguments
show_args(**more_args)    # Output: ONE TWO THREE
```

In [ ]:
# TODO 4.3 — Unpacking at the call site

def show_args(arg1, arg2, arg3="THREE"):
    print(arg1, arg2, arg3)


some_list = [0, 1, 2]
more_args = {"arg1": "ONE", "arg2": "TWO"}

# TODO 4.3-a: call show_args by UNPACKING some_list as positional args
show_args(*some_list)

# TODO 4.3-b: call show_args by UNPACKING more_args as keyword args
show_args(**more_args)


In [ ]:
# Quick check
import io, contextlib

buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    show_args(*some_list)
    show_args(**more_args)

out = buf.getvalue().splitlines()
assert out == ["0 1 2", "ONE TWO THREE"], out
print("OK: show_args(*some_list)   →", out[0])
print("OK: show_args(**more_args)  →", out[1])
print("Section 4.3 passed.")

## 5. Functions as First-Class Objects (Slides 18–25)

> *"In Python, functions are top-level objects just like strings, integers, or instances of a class."* — Slide 18

You can:

- Assign them to **variables**
- Pass them as **arguments** to other functions (callbacks)
- **Return** them from functions
- Attach **arbitrary attributes** to them dynamically

### 5.1 — Attaching Attributes & Passing Function Objects (Slide 19)

Because a function *is* an object, you can set attributes on it (`func.description = "..."`) and pass the whole thing to another function that calls or inspects it.

### Full code reference (Slide 19)

```python
# Function assignment and attribute manipulation
def my_function():
    print("The function was called!")


# Setting an attribute on the function object!
my_function.description = "A silly function"


# Passing the function object to another function
def another_function(func_object):
    print("Desc:", func_object.description)
    func_object()  # Actually executing it


another_function(my_function)
```

In [ ]:
# TODO 5.1 — Function attributes and first-class passing

def my_function():
    print("The function was called!")


# TODO 5.1-a: attach a `description` attribute to my_function with the string "A silly function"
my_function.description = "A silly function"


def another_function(func_object):
    # TODO 5.1-b: print "Desc:" followed by func_object.description
    print("Desc:", func_object.description)
    # TODO 5.1-c: actually call func_object()
    func_object()


another_function(my_function)


In [ ]:
# Quick check
assert my_function.description == "A silly function"
print(f"OK: my_function.description = {my_function.description!r}")

import io, contextlib
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    another_function(my_function)
out = buf.getvalue().splitlines()
assert out == ["Desc: A silly function", "The function was called!"], out
print("OK: another_function(my_function) printed both the description AND ran the function")
print("Section 5.1 passed.")

### 5.2 — Callbacks: Functions Passed as Arguments (Slides 20–21)

A **callback** is a function passed as an argument to another piece of code, with the expectation that the receiver will *call* it later.

> **Restaurant buzzer analogy.** You give the kitchen a buzzer (the callback). When your food is ready (the event), the kitchen *calls you back* by pushing it.

In `name_or_number` below, we pass a *list of test functions*. The receiver calls each one and uses `__name__` (a built-in attribute on every function object) to identify the winning rule.

### Full code reference (Slide 21)

```python
# Passing different test functions to a checker
def fizz(x): return x % 3 == 0
def buzz(x): return x % 5 == 0


def name_or_number(number, *tests):
    for test_func in tests:
        if test_func(number):
            return test_func.__name__
    return str(number)


print(name_or_number(3, fizz, buzz))   # fizz
print(name_or_number(5, fizz, buzz))   # buzz
```

In [ ]:
# TODO 5.2 — Callback functions

def fizz(x):
    # TODO 5.2-a: return True iff x is divisible by 3
    return x % 3 == 0


def buzz(x):
    # TODO 5.2-b: return True iff x is divisible by 5
    return x % 5 == 0


def name_or_number(number, *tests):
    # TODO 5.2-c: iterate over each test function in tests
    for test_func in tests:
        # TODO 5.2-d: if test_func(number) is True, return test_func.__name__
        if test_func(number):
            return test_func.__name__
    # TODO 5.2-e: if no test matched, return str(number)
    return str(number)


In [ ]:
# Quick check
assert name_or_number(3, fizz, buzz) == "fizz"
assert name_or_number(5, fizz, buzz) == "buzz"
assert name_or_number(7, fizz, buzz) == "7"
assert name_or_number(15, fizz, buzz) == "fizz"   # 15 hits fizz first (order matters)
print(f"OK: name_or_number(3, fizz, buzz)  = {name_or_number(3, fizz, buzz)!r}")
print(f"OK: name_or_number(5, fizz, buzz)  = {name_or_number(5, fizz, buzz)!r}")
print(f"OK: name_or_number(7, fizz, buzz)  = {name_or_number(7, fizz, buzz)!r}")
print(f"OK: name_or_number(15, fizz, buzz) = {name_or_number(15, fizz, buzz)!r}  (order: fizz wins)")
print("Section 5.2 passed.")

### 5.3 — Callbacks in the Standard Library: `sorted(key=…)` (Slide 23)

`list.sort(key=callable)` accepts *any* callable that, given an element, returns a sort key. Passing `str.lower` (the **method itself**, no parens) makes the sort case-insensitive.

### Full code reference (Slide 23)

```python
# Python's sort utilizing a function object
words = ["Zebra", "apple", "Yacht", "banana"]

# Standard sort uses ASCII values (Zebra < apple)
words.sort()
print(words)
# ['Yacht', 'Zebra', 'apple', 'banana']

# Passing str.lower function object as a callback
words.sort(key=str.lower)
print(words)
# ['apple', 'banana', 'Yacht', 'Zebra']
```

In [ ]:
# TODO 5.3 — Sorting with a callback

words_a = ["Zebra", "apple", "Yacht", "banana"]
words_b = list(words_a)   # an independent copy for the second sort

# TODO 5.3-a: sort words_a in place with the DEFAULT (ASCII) ordering
words_a.sort()

# TODO 5.3-b: sort words_b in place using str.lower as the key callback
#            Pass the method object itself — DO NOT call it (no parentheses)
words_b.sort(key=str.lower)

print("ASCII sort:        ", words_a)
print("Case-insensitive:  ", words_b)


In [ ]:
# Quick check
assert words_a == ['Yacht', 'Zebra', 'apple', 'banana']
assert words_b == ['apple', 'banana', 'Yacht', 'Zebra']
print("OK: ASCII order keeps capitals first ('Yacht' < 'apple')")
print("OK: str.lower as key folds case → 'apple' first")
print("Section 5.3 passed.")

### 5.4 — Monkey Patching: Assigning a Function to a Class at Runtime (Slides 24–25)

> *"Because functions are objects and methods are just functions bound to a class, you can assign an external function directly to a class attribute at runtime."*
>
> ⚠️ **Caution** — powerful, but should be used sparingly: it makes code harder to trace.

When you write `MyClass.dynamic_method = new_method`, Python stores the function in the class dict. On instance access, Python's descriptor protocol binds `self` automatically — `obj.dynamic_method()` works just like a normally defined method.

### Full code reference (Slide 25)

```python
# Monkey patching a method onto an existing class
class MyClass:
    def __init__(self, value):
        self.value = value


# An external function expecting 'self'
def new_method(self):
    print(f"The value is: {self.value}")


# Monkey patching: assign function to class
MyClass.dynamic_method = new_method


obj = MyClass(42)
obj.dynamic_method()   # The value is: 42
```

In [ ]:
# TODO 5.4 — Monkey patching

class MyClass:
    def __init__(self, value):
        self.value = value


def new_method(self):
    # TODO 5.4-a: print f"The value is: {self.value}"
    print(f"The value is: {self.value}")


# TODO 5.4-b: assign new_method to MyClass under the attribute name `dynamic_method`
MyClass.dynamic_method = new_method


obj = MyClass(42)
obj.dynamic_method()


In [ ]:
# Quick check
import io, contextlib

assert hasattr(MyClass, "dynamic_method"), "MyClass.dynamic_method should exist after monkey-patching"
assert MyClass.dynamic_method is new_method, "the patched attribute should BE new_method itself"

buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    MyClass(7).dynamic_method()
assert buf.getvalue().strip() == "The value is: 7"
print("OK: MyClass got dynamic_method at runtime; descriptor protocol bound self automatically")
print("Section 5.4 passed.")

## 6. The `__call__` Magic Method — Stateful Callables (Slides 26–27)

A standard function **loses state** between calls — every call starts fresh. A class with `__call__` lets an *instance* behave like a function while **retaining variables (state) via `self`** across calls.

### Full code reference (Slide 27)

```python
# A class implementing __call__
class CallCounter:
    def __init__(self):
        self.count = 0    # Persistent state

    def __call__(self, *args, **kwargs):
        self.count += 1
        print(f"Called {self.count} times.")


my_callable = CallCounter()
my_callable()   # Called 1 times.
my_callable()   # Called 2 times.
```

In [ ]:
# TODO 6 — CallCounter as a stateful callable

class CallCounter:
    def __init__(self):
        # TODO 6-a: initialise self.count = 0
        self.count = 0

    def __call__(self, *args, **kwargs):
        # TODO 6-b: increment self.count by 1
        self.count += 1
        # TODO 6-c: print f"Called {self.count} times."
        print(f"Called {self.count} times.")


In [ ]:
# Quick check
import io, contextlib

my_callable = CallCounter()
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    my_callable()
    my_callable()
    my_callable("extra", k=1)   # *args / **kwargs are accepted but ignored

out = buf.getvalue().splitlines()
assert out == ["Called 1 times.", "Called 2 times.", "Called 3 times."], out
assert my_callable.count == 3

# A fresh instance has its OWN state (proves the state lives on `self`, not on the class)
fresh = CallCounter()
assert fresh.count == 0
print("OK: my_callable retained state across 3 calls →", out)
print("OK: a fresh CallCounter has its own count = 0 (state lives on self, not the class)")
print("Section 6 passed.")

## 7. Context Managers — Managing External State (Slides 28–31)

> **The Golden Rule.** Any time we work with an external resource (file, network socket, database), we must ensure it is **properly closed**, even if an exception occurs.

The `with` statement guarantees that. Under the hood it's just two dunder methods:

| Step | What Python does                                                |
| ---- | --------------------------------------------------------------- |
| 1    | Call `obj.__enter__()`                                          |
| 2    | Bind its return value to the variable after `as`                |
| 3    | Run the indented block                                          |
| 4    | Call `obj.__exit__(exc_type, exc_val, exc_tb)`                  |
| 5    | `__exit__` cleans up the resource (closes the file, etc.)       |

### 7.1 — Reading a File Safely with `with` (Slide 29)

`Path("...").open()` returns a file object — itself a context manager. The `with` block ensures `.close()` runs even if the loop body raises.

### Full code reference (Slide 29)

```python
# Safe file reading with the 'with' statement
from pathlib import Path

# 'with' guarantees the file is closed automatically
with Path("sample_data.txt").open('r', encoding='utf-8') as file:
    for line in file:
        print(line.strip())
# At this point, the file is safely closed.
```

In [ ]:
# TODO 7.1 — Read sample_data.txt safely with `with`
from pathlib import Path

# TODO 7.1-a: open sample_data.txt in 'r' mode with utf-8 encoding via Path(...).open()
#             and bind the file handle to `file`
with Path("sample_data.txt").open('r', encoding='utf-8') as file:
    # TODO 7.1-b: iterate over `file` and print line.strip() for each
    for line in file:
        print(line.strip())


In [ ]:
# Quick check — confirm the file got closed after the with-block.
# We rebuild the file object here just to assert that the with-block finished cleanly.
with Path("sample_data.txt").open('r', encoding='utf-8') as f:
    lines = [line.rstrip() for line in f]

assert f.closed, "the with-block should have closed the file automatically"
assert lines == ["first line", "second line", "third line"], lines
print(f"OK: read {lines}")
print("OK: f.closed is True — __exit__ closed the handle for us")
print("Section 7.1 passed.")

### 7.2 — Building a Custom Context Manager (Slide 31)

Any class with `__enter__` / `__exit__` is a context manager. The slide-31 `StringJoiner`:

- `__enter__` prints `"Entering context"` and returns `self.strings` (so the loop body can `.append` directly).
- `__exit__` prints the joined result when the `with` block ends — even if an exception was raised inside.

Note: `__enter__` returns `self.strings` (a list), so inside the `with` block, `joiner` *is* that list — that's why `joiner.append(...)` works.

### Full code reference (Slide 31)

```python
# A class with __enter__ and __exit__
class StringJoiner:
    def __init__(self):
        self.strings = []

    def __enter__(self):
        print("Entering context")
        return self.strings

    def __exit__(self, exc_type, exc_val, exc_tb):
        print("".join(self.strings))


with StringJoiner() as joiner:
    joiner.append("Hello")
    joiner.append(" World!")
# Output:
#   Entering context
#   Hello World!
```

In [ ]:
# TODO 7.2 — StringJoiner

class StringJoiner:
    def __init__(self):
        # TODO 7.2-a: initialise self.strings as an empty list
        self.strings = []

    def __enter__(self):
        # TODO 7.2-b: print "Entering context"
        print("Entering context")
        # TODO 7.2-c: return self.strings so the `as` variable IS the list
        return self.strings

    def __exit__(self, exc_type, exc_val, exc_tb):
        # TODO 7.2-d: print the joined result of self.strings
        # Hint: "".join(self.strings)
        print("".join(self.strings))


In [ ]:
# Quick check
import io, contextlib

buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    with StringJoiner() as joiner:
        joiner.append("Hello")
        joiner.append(" World!")

out = buf.getvalue().splitlines()
assert out == ["Entering context", "Hello World!"], out
print("OK: StringJoiner output:")
for line in out:
    print("   ", line)
print("Section 7.2 passed.")

## 8. Case Study — K-NN Train/Test Split, the Functional Way (Slides 32–40)

### Continuity with Lecture 5

In Lecture 5 §7 we built `CountingDealingPartition` — a *class* that subclassed `list[dict]` and dispatched each appended item to `_training` or `_testing` based on `counter % 10 < 8`. It worked, but it baked the rule into the class.

This time we follow slide 32's prescription:

> *"Move away from strictly stateful classes toward functional composition."*

We separate the **rule** (a tiny standalone function: "is this index training data?") from the **loop** (a higher-order `partition` that takes the rule as a callback). Different splits = different rule cartridges, **same** machine.

```
Sample Objects  →  (Partition Function + Rule callback)  →  Training Set & Test Set
```

### 8.1 — Functional Splitting Rules (Slides 34–35)

Three split ratios encoded as one-line predicates. Each one returns `True` if the indexed sample should go to **training**.

| Function       | Ratio   | Training when…       |
| -------------- | ------- | -------------------- |
| `training_80`  | 80 / 20 | `index % 5 != 0`     |
| `training_75`  | 75 / 25 | `index % 4 != 0`     |
| `training_67`  | 67 / 33 | `index % 3 != 0`     |

### Full code reference (Slide 35)

```python
# Returns True if it should be Training Data
def training_80(sample, index: int) -> bool:
    return index % 5 != 0

def training_75(sample, index: int) -> bool:
    return index % 4 != 0

def training_67(sample, index: int) -> bool:
    return index % 3 != 0
```

In [ ]:
# TODO 8.1 — Three split rules as standalone functions

def training_80(sample, index: int) -> bool:
    # TODO 8.1-a: return True iff index % 5 != 0  (i.e. NOT every 5th item)
    return index % 5 != 0


def training_75(sample, index: int) -> bool:
    # TODO 8.1-b: return True iff index % 4 != 0
    return index % 4 != 0


def training_67(sample, index: int) -> bool:
    # TODO 8.1-c: return True iff index % 3 != 0
    return index % 3 != 0


In [ ]:
# Quick check — index 0 always goes to TESTING (0 % anything == 0).
sample = {"id": "A"}
assert training_80(sample, 0) is False and training_80(sample, 1) is True
assert training_75(sample, 0) is False and training_75(sample, 1) is True
assert training_67(sample, 0) is False and training_67(sample, 1) is True

# Spot-check on a 10-item run for the 80/20 rule: indices 0 and 5 go to test.
flags_80 = [training_80(None, i) for i in range(10)]
assert flags_80 == [False, True, True, True, True, False, True, True, True, True]
print(f"OK: training_80 over 0..9 → {flags_80}")
print("Section 8.1 passed.")

### 8.2 — Higher-Order `partition` (Two-Pass) (Slides 36–39)

> *"Think of the higher-order function as a machine with an empty slot for a cartridge."*

`partition(samples, rule)` takes the data **and** the rule. Internally it runs *two* list comprehensions — one for the trues, one for the falses.

### Full code reference (Slide 39)

```python
# Functional list generation with higher-order partitioning
from typing import Callable, Iterable

def partition(
    samples: Iterable,
    rule: Callable[[object, int], bool]
):
    training = [
      s for i, s in enumerate(samples)
      if rule(s, i)]
    test = [
      s for i, s in enumerate(samples)
      if not rule(s, i)]
    return training, test


train, test = partition(my_data, training_80)
```

In [ ]:
# TODO 8.2 — Higher-order partition (two-pass)
from typing import Callable, Iterable


def partition(
    samples: Iterable,
    rule: Callable[[object, int], bool],
):
    # TODO 8.2-a: list comprehension — keep s where rule(s, i) is True
    training = [
        s for i, s in enumerate(samples)
        if rule(s, i)
    ]
    # TODO 8.2-b: list comprehension — keep s where rule(s, i) is False
    test = [
        s for i, s in enumerate(samples)
        if not rule(s, i)
    ]
    # TODO 8.2-c: return both lists as a tuple (training, test)
    return training, test


In [ ]:
# Quick check — the 80/20 rule on 10 items.
my_data = [{"id": c} for c in "ABCDEFGHIJ"]
train, test = partition(my_data, training_80)

train_ids = [d["id"] for d in train]
test_ids = [d["id"] for d in test]

# index 0 ('A') and index 5 ('F') hit the % 5 == 0 branch → TEST; everyone else → TRAIN
assert train_ids == list("BCDEGHIJ"), train_ids
assert test_ids == list("AF"), test_ids
print(f"OK: training_80 → train = {train_ids}")
print(f"OK: training_80 → test  = {test_ids}")

# Swap the cartridge — same machine, different output.
train_75, test_75 = partition(my_data, training_75)
assert [d["id"] for d in test_75] == list("AEI")   # indices 0,4,8
print(f"OK: training_75 → test  = {[d['id'] for d in test_75]}")
print("Section 8.2 passed.")

### 8.3 — Optimizing to a Single Pass (Slide 40)

The two-pass version evaluates `rule()` **twice per item** and walks the data **twice**. For massive datasets that's wasteful. The one-pass version walks once and calls `rule()` once per item.

| Variant       | Loops | `rule()` calls | Code                  |
| ------------- | ----- | -------------- | --------------------- |
| `partition`   | 2     | 2 × N          | concise comprehensions |
| `partition_1` | 1     | 1 × N          | explicit loop          |

Same input → same output: we'll assert it directly.

### Full code reference (Slide 40)

```python
# Optimized single-pass approach
def partition_1(samples, rule):
    training = []
    testing = []
    for i, s in enumerate(samples):
        if rule(s, i):
            training.append(s)
        else:
            testing.append(s)
    return training, testing
```

In [ ]:
# TODO 8.3 — partition_1: single-pass version

def partition_1(samples, rule):
    # TODO 8.3-a: initialise training and testing as empty lists
    training = []
    testing = []
    # TODO 8.3-b: enumerate over samples once
    for i, s in enumerate(samples):
        # TODO 8.3-c: if rule(s, i) is True append to training, else append to testing
        if rule(s, i):
            training.append(s)
        else:
            testing.append(s)
    # TODO 8.3-d: return (training, testing)
    return training, testing


In [ ]:
# Quick check — the one-pass and two-pass versions MUST produce the same split.
my_data = [{"id": c} for c in "ABCDEFGHIJKLMNOP"]   # 16 items

for rule in (training_80, training_75, training_67):
    a_train, a_test = partition(my_data, rule)
    b_train, b_test = partition_1(my_data, rule)
    assert a_train == b_train, f"{rule.__name__}: training mismatch"
    assert a_test == b_test, f"{rule.__name__}: testing mismatch"
    print(f"OK: {rule.__name__:<12} → identical split for both partition variants "
          f"({len(a_train)} train / {len(a_test)} test)")

print("Section 8.3 passed.")

### 8.4 — Bridge to Lecture 5: Same Split, Two Paradigms

Lecture 5 §7.4 produced the 80/20 split with a *class* (`CountingDealingPartition`) — its rule was hard-coded inside `append()`:

```python
if self.counter % d < n:   # n=8, d=10
    self._training.append(item)
else:
    self._testing.append(item)
```

That's `index % 10 < 8` — the rule says "every 10th block: items 0–7 go to train, 8–9 to test." Our `training_80` (`index % 5 != 0`) uses a different period (5 instead of 10), but on a 10-item run the *count* lands the same way: 8 train, 2 test. The point isn't that they produce identical lists — it's that the **rule lives in two completely different places**:

| Lecture 5 (OOP)                              | Lecture 6 (Functional)                          |
| -------------------------------------------- | ----------------------------------------------- |
| Rule baked into `CountingDealingPartition.append` | Rule = a free function (`training_80`)          |
| Loop = `for it in items: cdp.append(it)`     | Loop = `partition(my_data, training_80)`        |
| To change the ratio: subclass + override     | To change the ratio: pass a different cartridge |

The cell below recreates Lecture 5's dealing logic locally and shows that for `(n,d) = (8,10)` it produces **the same train/test counts** as `partition(..., training_80)` on a 10-item run.

In [ ]:
# Bridge demo — re-implement Lecture 5's CountingDealingPartition logic with the
# Lecture 6 partition() machine, by writing the (n,d) rule as a tiny callback.

# Lecture 5 rule, expressed as a Lecture 6-style callback
def training_n_of_d(n: int, d: int):
    """Lecture 5's CountingDealingPartition rule, as a cartridge factory."""
    def rule(sample, index: int) -> bool:
        return index % d < n
    rule.__name__ = f"training_{n}_of_{d}"
    return rule


items = [{"id": c} for c in "ABCDEFGHIJ"]   # 10 items, same as Lecture 5 §7.4

l5_rule = training_n_of_d(8, 10)            # Lecture 5's (n=8, d=10)
l5_train, l5_test = partition_1(items, l5_rule)

print(f"Lecture-5 rule (n=8, d=10) via partition_1:")
print(f"  train: {[d['id'] for d in l5_train]}")
print(f"  test:  {[d['id'] for d in l5_test]}")

# Sanity: matches Lecture 5 §7.4 exactly — A..H to train, I, J to test.
assert [d['id'] for d in l5_train] == list("ABCDEFGH")
assert [d['id'] for d in l5_test]  == list("IJ")
print("\nOK: same split as Lecture 5 §7.4 — but the rule is now a swappable callback,")
print("    not a hard-coded branch inside an OOP class.")
print("Section 8.4 passed.")

## 9. Summary (Slide 41)

> **Python elegantly marries OOP and Functional paradigms.**
>
> By utilizing built-in functions, understanding `*args`/`**kwargs`, and leveraging first-class function objects (callbacks), we can build highly expressive, decoupled systems without rigid class hierarchies.

### What you practiced

| Slide   | Topic                                        | What you built                                       |
| ------- | -------------------------------------------- | ---------------------------------------------------- |
| 4       | `len()` ↔ `__len__`                          | `CustomSequence`                                     |
| 5–7     | `reversed()` fallback + override             | `CardDeck`, `FunkyBackwards`                         |
| 8       | `enumerate(start=…)`                         | line-numbered file reader                            |
| 9–11    | "No method overloading" — defaults           | `add`, `mandatory_and_optional`                      |
| 12–13   | Mutable default trap + cure                  | `bad_append` vs `good_append`                        |
| 14–17   | `*args`, `**kwargs`, call-site unpacking     | `get_pages`, `connect_to_server`, `show_args`        |
| 18–19   | Functions as first-class objects             | function attributes + passing                        |
| 20–23   | Callbacks                                    | fizz/buzz, `sort(key=str.lower)`                     |
| 24–25   | Monkey patching                              | `MyClass.dynamic_method`                             |
| 26–27   | `__call__` — stateful callables              | `CallCounter`                                        |
| 28–31   | Context managers                             | `with` + custom `StringJoiner`                       |
| 32–40   | Functional K-NN partition (continuity)       | `training_80/75/67`, `partition`, `partition_1`      |

### Cleanup

In [ ]:
# Tidy up the sample file we created in Section 2.
from pathlib import Path
Path("sample_data.txt").unlink(missing_ok=True)
print("Cleaned up sample_data.txt")

Great work! 🎯 You've now seen Python's "two sides of the same coin" — every higher-order, functional pattern you used here is powered by an OOP dunder method underneath.